# Workflow 1

This notebook is the first workflow as descrbed in this google document - https://docs.google.com/document/d/1qbmAjRa2V-anxj63YFFHdnovbY_U9MRi0uMAa4ZoCNg/edit.

The notebook merges the predictions (adjacent tiles) from SAM and does accuracy assessment.

In [1]:
#import sys
#sys.path.append('../')
from utils import *

# import required libraries
import geopandas as gpd
import pandas as pd
import numpy as np
import geoplanar
import os, glob, math, json
from matplotlib import pyplot as plt
from copy import deepcopy
from rtree import index
from geopandas.tools import sjoin
from shapely.geometry import box
from concurrent.futures import ThreadPoolExecutor
import time
from datetime import datetime
import rasterio
from rasterstats import zonal_stats
from copy import deepcopy

today = datetime.now().strftime("%y%m%d")

# change the working directory
main_dir = r'D:\2212_PlanetOrtho_FieldBoundary\AAG\vector\230606_Fragments'
out_dir = r'D:\2212_PlanetOrtho_FieldBoundary\ForPaper\vector'
#gt_infile = r"D:\2212_PlanetOrtho_FieldBoundary\AAG\vector\GroundTruth\231019_GroundTruth_FieldBoundaries_PT_V2.gpkg"
os.chdir(main_dir)

## Merge adjacent tiles

In [2]:
%%time

# area threshold to remove outliers (absolute number at lower end and percentage of tile area for upper end)
AREA_THRESHOLD = [15, .6, 100, 15000]
# boundary overlap threshold for touching polygons in adjacent tiles
OVERLAP_THRESHOLD = 85

# create folder
out_folder = os.path.join(out_dir, f'{today}_Workflow1_Output_v1')
if not os.path.exists(out_folder):
    os.mkdir(out_folder)

# define a dataframe to store metrics
main_df = pd.DataFrame([], columns=['name', 'detected', 'detection_percentage', 'iou', 'precision', 'recall', 'f1score'])

folders = glob.glob('T*/')
for n, folder in enumerate(folders):
    if ('Enhanced' not in folder) & (not 'Output' in folder):
        print(folder, f'{n}/{len(folders)}...')
        t1 = time.time()

        # read files as shapefile all three checkpoints separately
        gdf_vitb = pd.concat(read_file_with_name(file) for file in glob.glob(f'{folder}/*_vit_b_*.gpkg'))
        gdf_vith = pd.concat(read_file_with_name(file) for file in glob.glob(f'{folder}/*_vit_l_*.gpkg'))
        gdf_vitl = pd.concat(read_file_with_name(file) for file in glob.glob(f'{folder}/*_vit_h_*.gpkg'))

        for test_gdf in [gdf_vitb, gdf_vith, gdf_vitl]:
            test_gdf = get_row_col(test_gdf)

            # apply area threshold to remove outliers
            row_size = test_gdf['row_val'].astype(int).max()
            col_size = test_gdf['col_val'].astype(int).max()
            tile_width = 3072 * 0.8 / col_size
            tile_height = 3072 * 0.8 / row_size
            tile_area = tile_width * tile_height

            # remove smaller and larger polys before merging adjacent ones
            test_gdf = test_gdf.loc[(test_gdf.area > AREA_THRESHOLD[0]) * (test_gdf.area < AREA_THRESHOLD[1] * tile_area)]
           
            # merge edge polygons from adjacent layers
            test_gdf = merge_adjacent_tiles(test_gdf, folder, OVERLAP_THRESHOLD, verbose=False)

            # apply another area based filtering after merging
            test_gdf = test_gdf.loc[
                (test_gdf.area > AREA_THRESHOLD[2]) * (test_gdf.area < AREA_THRESHOLD[3])
            ]
            
            # add a unique identifier for reference
            test_gdf = test_gdf.drop('index', axis=1).explode(index_parts=True).reset_index(drop=True)
            test_gdf['ID_vit'] = test_gdf.index

            outfile1 = os.path.join(out_folder, get_filename(test_gdf))
            test_gdf['area'] = test_gdf.area
            #test_gdf.to_file(outfile1)
            
            # simplify the edges and make them rectangular-ish
            test_gdf['geometry'] = test_gdf['geometry'].apply(
                make_rectangularish, simplify_tolerance=2, buffer_distance=1
            )
            
            # take care of spillovers
            test_gdf = geoplanar.trim_overlaps(test_gdf, largest=False)
            
            # export the processed file
            test_gdf.to_file(outfile1)
            
            
        t2 = time.time()
        print(f'{folder} folder processed in {(t2 - t1) / 60} mins.')

T1_12by12\ 0/32...


C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\gdalenv\Lib\site-packages\pyogrio\geopandas.py:523: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  has_z_arr = geometry[geometry.notna() & (~geometry.is_empty)].has_z


T1_12by12\ folder processed in 0.5779720664024353 mins.
T1_3by3\ 2/32...
T1_3by3\ folder processed in 0.86051477988561 mins.
T1_4by4\ 4/32...


C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\gdalenv\Lib\site-packages\pyogrio\geopandas.py:523: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  has_z_arr = geometry[geometry.notna() & (~geometry.is_empty)].has_z


T1_4by4\ folder processed in 1.1031178514162698 mins.
T1_6by6\ 6/32...


...

T2_6by6\ folder processed in 0.7874899784723918 mins.
T3_12by12\ 16/32...


C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\gdalenv\Lib\site-packages\pyogrio\geopandas.py:523: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  has_z_arr = geometry[geometry.notna() & (~geometry.is_empty)].has_z
C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\gdalenv\Lib\site-packages\pyogrio\geopandas.py:523: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has chang

T3_12by12\ folder processed in 0.8278650999069214 mins.
T3_3by3\ 18/32...
T3_3by3\ folder processed in 1.1034427285194397 mins.
T3_4by4\ 20/32...
T3_4by4\ folder processed in 1.1326805313428243 mins.
T3_6by6\ 22/32...
T3_6by6\ folder processed in 1.2076151529947916 mins.
T4_12by12\ 24/32...
T4_12by12\ folder processed in 0.6722889582316081 mins.
T4_3by3\ 26/32...
T4_3by3\ folder processed in 1.1243682622909545 mins.
T4_4by4\ 28/32...


C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\gdalenv\Lib\site-packages\pyogrio\geopandas.py:523: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  has_z_arr = geometry[geometry.notna() & (~geometry.is_empty)].has_z


T4_4by4\ folder processed in 1.109943159421285 mins.
T4_6by6\ 30/32...
T4_6by6\ folder processed in 1.1060492237408956 mins.
CPU times: total: 4min 36s
Wall time: 14min 25s


## ~On the exported shapefile, run zonal statistics on NDVI layer (or use Python's exactextract package)~
~After running zonal stats, filter out polygons using NDVI values and then do accuracy assessment by comparing to GT data.~<br/>
NO NEED TO DO THIS ANYMORE, NDVI ZONAL STATS STEP HAS BEEN INTEGRATED IN THE CODE CHUNK BELOW, INSIDE THE LOOP.

In [13]:
# define input GT file
gt_file = r"D:\2212_PlanetOrtho_FieldBoundary\ForPaper\vector\GroundTruth\240805_GroundTruth_FieldBoundaries_PT_V2.shp"
ndvi_file = r"D:\2212_PlanetOrtho_FieldBoundary\ForPaper\raster\NDVI_max.tif"

# get the imput file names
out_folder = r"D:\\2212_PlanetOrtho_FieldBoundary\\ForPaper\\vector\\240814_Workflow1_Output_v1"
files_list = glob.glob(f"{out_folder}/*Merged.gpkg")

In [14]:
%%time

NDVI_THRESHOLD = 0.1
COMPACTNESS_THRESHOLD = 0.5

# create an empty DF to store metrics
metrics_df = pd.DataFrame(columns=['File', 'Precision', 'Recall', 'F1-Score', 'Detection', 'IoU'])

# loop through all the files and caculate metrics
for n, file in enumerate(files_list):
    print(os.path.split(file)[-1], f'{n}/{len(files_list)}..')
    
    """
    prepare the ground truth shapefile
    """
    gt_gdf = gpd.read_file(gt_file)
    gt_gdf['ID_gt'] = gt_gdf.index

    # add new cols in the ground truth gdf to calculate metrics later
    for new_col in ['max_overlap_id', 'iou', 'precision', 'recall', 'f1score']:
        gt_gdf[new_col] = -1 

    # remove empty geometries
    gt_gdf = gt_gdf.loc[
        (gt_gdf.geom_type.isin(['Polygon', 'MultiPolygon'])) & (gt_gdf['NDVI_mean'] > NDVI_THRESHOLD)
    ]

    """
    read the predicted data
    """
    # read the predicted file
    pred_gdf = gpd.read_file(file)
    
    # apply area threshold otherwise zonal stats fails (small polygons got created at geoplanar overlap trim stage)
    pred_gdf = pred_gdf.loc[pred_gdf.area > 15]
    
    """
    apply NDVI based filter
    """
    # apply filter using NDVI value
    pred_gdf['NDVI_mean'] = [
        element['mean'] 
        for element in zonal_stats(pred_gdf, ndvi_file, stats=['mean'])
    ]
    pred_gdf = pred_gdf.loc[pred_gdf['NDVI_mean'] > NDVI_THRESHOLD]

    """
    extract meaningful polygons using fill proportion
    """
    bounding_boxes = pred_gdf.geometry.apply(lambda geom: geom.minimum_rotated_rectangle)
    pred_gdf = pred_gdf[pred_gdf.area / bounding_boxes.area >= COMPACTNESS_THRESHOLD]
    
    """
    export the cleaned predicted file
    """
    pred_gdf.to_file(file.replace('.gpkg', '_V2.gpkg'))
    
    """
    calculate tota area based metrics
    """
    intersection_area = gt_gdf.dissolve().intersection(pred_gdf.dissolve()).area[0]
    union_area = gt_gdf.dissolve().union(pred_gdf.dissolve()).area[0]

    precision = round(intersection_area / pred_gdf.dissolve().area[0], 3)
    recall = round(intersection_area / gt_gdf.dissolve().area[0], 3)
    f1_score = round(2 * (precision * recall) / (precision + recall), 3)

    """
    calculate number of polygons detected
    """
    # Create spatial index for test layer for faster processing
    # can't craete this inside the function because it is unnecessary time wastage
    spatial_index = index.Index()
    for i, geometry in enumerate(pred_gdf.geometry):
        if len(geometry.bounds) > 0:
            spatial_index.insert(i, geometry.bounds)

    # for each polygon in the ground truth, find polygon in the test file that has IoU greater than 0.5
    gt_gdf = gt_gdf.apply(lambda row: wflow1_iou_threshold_metrics(row, gt_gdf, deepcopy(pred_gdf), spatial_index,
                                                                  threshold=0.5), axis=1)

    # calculate the precision for this test layer (how many polygons are identified)
    detection_percentage = round(100 * gt_gdf.loc[gt_gdf['max_overlap_id'] != -1].shape[0] / gt_gdf.shape[0], 3)
    detected_iou = round(gt_gdf.loc[gt_gdf['max_overlap_id'] != -1, 'iou'].mean(), 3)

    metrics_dict = {
            'File': [os.path.split(file)[-1]],
            'Precision': [precision],
            'Recall': [recall],
            'F1-Score': [f1_score],
            'Detection': [detection_percentage],
            'IoU': [detected_iou]
        }

    metrics_df = pd.concat([
        metrics_df, 
        pd.DataFrame(metrics_dict)
    ], axis=0)

    metrics_df.to_csv(f"D:/2212_PlanetOrtho_FieldBoundary/ForPaper/tables/{today}_Workflow1Metrics_v1.csv", 
                      index=False)

T1_12by12_10_vit_b_segmented_Workflow1Merged.gpkg 0/96..


<timed exec>:93: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


T1_12by12_10_vit_h_segmented_Workflow1Merged.gpkg 1/96..
T1_12by12_10_vit_l_segmented_Workflow1Merged.gpkg 2/96..
T1_12by12_Enhanced_10_vit_b_segmented_Workflow1Merged.gpkg 3/96..
T1_12by12_Enhanced_10_vit_h_segmented_Workflow1Merged.gpkg 4/96..
T1_12by12_Enhanced_4_vit_l_segmented_Workflow1Merged.gpkg 5/96..
T1_3by3_1_vit_b_segmented_Workflow1Merged.gpkg 6/96..
T1_3by3_1_vit_h_segmented_Workflow1Merged.gpkg 7/96..
T1_3by3_1_vit_l_segmented_Workflow1Merged.gpkg 8/96..
T1_3by3_Enhanced_1_vit_b_segmented_Workflow1Merged.gpkg 9/96..
T1_3by3_Enhanced_1_vit_h_segmented_Workflow1Merged.gpkg 10/96..
T1_3by3_Enhanced_1_vit_l_segmented_Workflow1Merged.gpkg 11/96..
T1_4by4_1_vit_b_segmented_Workflow1Merged.gpkg 12/96..
T1_4by4_1_vit_h_segmented_Workflow1Merged.gpkg 13/96..
T1_4by4_1_vit_l_segmented_Workflow1Merged.gpkg 14/96..
T1_4by4_Enhanced_1_vit_b_segmented_Workflow1Merged.gpkg 15/96..
T1_4by4_Enhanced_1_vit_h_segmented_Workflow1Merged.gpkg 16/96..
T1_4by4_Enhanced_1_vit_l_segmented_Workflow